# AQUA-STRAT — Daily LSTM Inference on Fresh Copernicus Data (v1)

**What this does:** uses your already-trained `aqua_strat_lstm_v5.keras` (NO retraining) on the latest **real** Copernicus ocean physics
(SST, salinity, u/v currents, SSH) to produce **real model predictions for every grid cell, for every day** —
the last ~3 weeks (analysis physics) plus the next ~10 days (Copernicus forecast physics).

**How to run:** `Runtime → Run all`. CPU runtime is enough (no GPU needed). Takes ~10–25 min.
You will be asked once for your **Copernicus Marine username + password** (free account at marine.copernicus.eu).

**Files it needs** (it searches your Drive folder `MyDrive/Aquastrat` automatically — **including zips and zips inside zips**, e.g. your `lstm_v5_final-….zip` inside another zip):
`grid_master_full.csv`, `aqua_strat_lstm_v5.keras`, `predicted_hotspots_v5.csv`, and your **original 2025 Copernicus NetCDF files** (needed to rebuild the exact training normalisation).
If something is not in Drive, the notebook asks for it **one file at a time** — pick the file itself, or any `.zip` that contains it.

**At the end:** a file `aquastrat_daily_predictions_<date>.zip` downloads automatically — send that zip back for integration.


## Step 1 — Install, mount Drive, imports

In [ ]:
!pip -q install copernicusmarine
from google.colab import drive
drive.mount("/content/drive")

import os, glob, zipfile, shutil, warnings, getpass, datetime as dt
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xarray as xr
import tensorflow as tf
import copernicusmarine as cm

import re
MYDRIVE = "/content/drive/MyDrive"
# your project folder(s) on Drive — any capitalisation (Aquastrat / AquaStrat / AQUASTRAT) is found automatically
BASES = [os.path.join(MYDRIVE, d) for d in sorted(os.listdir(MYDRIVE)) if d.lower() == "aquastrat"]
BASE = os.path.join(MYDRIVE, "Aquastrat") if os.path.isdir(os.path.join(MYDRIVE, "Aquastrat")) else (BASES[0] if BASES else os.path.join(MYDRIVE, "Aquastrat"))
os.makedirs(BASE, exist_ok=True)
if BASE not in BASES: BASES.insert(0, BASE)
SCRATCH = "/content/aqua_strat_nc_scratch"      # .nc extracted from zips (original notebook logic)
NESTED = "/content/aqua_nested"                 # zips found INSIDE zips are unpacked here
UPLOAD_DIR = "/content/aqua_uploads"            # anything you upload goes here
for d in (SCRATCH, NESTED, UPLOAD_DIR): os.makedirs(d, exist_ok=True)
print("Project folder(s) on Drive:", BASES)

def expand_zip(zp, out_root, want=(".csv", ".keras", ".zip"), depth=0):
    # Unpack the wanted files from a zip and from zips inside it (any depth). Sentinel/.SAFE archives are skipped.
    low = os.path.basename(zp).lower()
    if depth > 6 or ".safe" in low or "sentinel" in low: return
    dest = os.path.join(out_root, re.sub(r"[^A-Za-z0-9_.-]", "_", os.path.splitext(os.path.basename(zp))[0]) + f"__{depth}")
    try:
        zf = zipfile.ZipFile(zp)
    except Exception as e:
        print("  [skip] not a readable zip:", os.path.basename(zp), "|", e); return
    with zf:
        for m in zf.namelist():
            if m.endswith("/") or not m.lower().endswith(want): continue
            safe = os.path.normpath(m)
            if safe.startswith("..") or os.path.isabs(safe): continue
            t = os.path.join(dest, safe); os.makedirs(os.path.dirname(t), exist_ok=True)
            if not os.path.exists(t):
                with zf.open(m) as src, open(t, "wb") as dst: shutil.copyfileobj(src, dst, length=1 << 20)
            if t.lower().endswith(".zip"):
                expand_zip(t, out_root, want, depth + 1)

def search_roots():
    return BASES + [NESTED, UPLOAD_DIR]

PAST_DAYS = 36      # history pulled (14-day window + ~3 weeks of daily predictions)
FUTURE_DAYS = 10    # Copernicus forecast physics ahead (clipped automatically to what exists)
WINDOW, HORIZON = 14, 1
DYNAMIC_VARS = ["thetao", "so", "uo", "vo", "zos"]
REGIONS = ["AS", "BOB", "IO"]
TODAY = pd.Timestamp(dt.date.today())
print("BASE exists:", os.path.exists(BASE), "| today:", TODAY.date(), "| TF", tf.__version__)

## Step 2 — Locate grid master, trained model and v5 predictions

In [ ]:
from google.colab import files
NEEDED = ["grid_master_full.csv", "aqua_strat_lstm_v5.keras", "predicted_hotspots_v5.csv"]

# 1) open every zip in your Drive project folder, including zips inside zips (e.g. lstm_v5_final-....zip inside another zip)
for root in BASES:
    for zp in glob.glob(root + "/**/*.zip", recursive=True):
        expand_zip(zp, NESTED)

def find(name):
    hits = glob.glob("/content/" + name)
    for root in search_roots():
        hits += glob.glob(os.path.join(root, "**", name), recursive=True)
    return sorted(set(hits), key=len)[0] if hits else None

def save_upload(up, want):
    for fn, data in up.items():
        dst = os.path.join(UPLOAD_DIR, os.path.basename(fn))
        with open(dst, "wb") as f: f.write(data)
        if fn.lower().endswith(".zip"): expand_zip(dst, UPLOAD_DIR, want=want)

def upload_one(name):
    # Asks for ONE file at a time: the file itself OR any .zip that contains it (zip inside zip is fine).
    while True:
        print(f"\n>>> Please upload: {name}\n    (pick the file, or the .zip that has it inside - even a zip inside a zip). Cancel = stop.")
        up = files.upload()
        if not up: return None
        save_upload(up, (".csv", ".keras", ".zip", ".nc"))
        p = find(name)
        if p: print("    found:", p); return p
        print(f"    {name} was not inside that upload - try again.")

paths = {}
for n in NEEDED:
    paths[n] = find(n) or upload_one(n)
for n, p in paths.items():
    print(f"{n:28s} -> {p}")
    assert p, f"{n} is required"

MASTER_GRID_PATH = paths["grid_master_full.csv"]
master = pd.read_csv(MASTER_GRID_PATH)
master["region"] = master["region"].astype(str).str.upper()
model = tf.keras.models.load_model(paths["aqua_strat_lstm_v5.keras"])
v5 = pd.read_csv(paths["predicted_hotspots_v5.csv"])
print("Model input:", model.input_shape, "| v5 cells:", len(v5))
print(v5.groupby("region").size())

## Step 3 — Rebuild the EXACT training normalisation
The training notebook normalised features with mean/std from the first 70% of the 2025 data but did not save them.
This step re-runs **your original notebook's own data-loading code (copied unchanged)** on your 2025 NetCDF files to recover them,
then **verifies** them by reproducing your saved `predicted_hotspots_v5.csv`.

In [ ]:
# Your 2025 Copernicus NetCDF files (the ones the LSTM was trained on) are looked for in Drive, in zips, and in zips inside zips.
def nc_available():
    n = 0
    for root in search_roots():
        n += len(glob.glob(root + "/**/*.nc", recursive=True))
        for zp in glob.glob(root + "/**/*.zip", recursive=True):
            low = os.path.basename(zp).lower()
            if ".safe" in low or "sentinel" in low: continue
            try:
                with zipfile.ZipFile(zp) as zf: n += sum(m.lower().endswith(".nc") for m in zf.namelist())
            except Exception: pass
    return n
n_nc = nc_available()
print("NetCDF files available (direct + inside zips):", n_nc)
if n_nc == 0:
    print("No 2025 NetCDF files found in Drive. Upload them ONE AT A TIME (each .nc, or a .zip containing them).")
    print("Tip: big files are faster if you copy them into your Drive 'Aquastrat' folder and re-run this cell.")
    while True:
        up = files.upload()
        if not up: break
        save_upload(up, (".nc", ".zip"))
        print("Received so far:", nc_available(), "NetCDF file(s). Upload the next one, or press Cancel when done.")

In [ ]:
# ---- copied from AQUA_STRAT_LSTM_v5 (Step 3: discover NetCDF data); only change: also searches unpacked/uploaded zips ----
DYNAMIC_VARS = ["thetao", "so", "uo", "vo", "zos"]
REGIONS = ["AS", "BOB", "IO"]

def all_files(base):
    out = []
    for root, dirs, files in os.walk(base):
        for f in files:
            out.append(os.path.join(root, f))
    return out

all_paths = [p for r in search_roots() for p in all_files(r)]
direct_nc = [p for p in all_paths if p.lower().endswith(".nc")]
zip_paths = [p for p in all_paths if p.lower().endswith(".zip")]

print("Direct NC files:", len(direct_nc))
print("ZIP files:", len(zip_paths))

for p in direct_nc:
    print("NC:", p)
for p in zip_paths:
    print("ZIP:", p)

In [ ]:
def extract_nc_from_zip(zip_path, out_root):
    out_dir = os.path.join(out_root, os.path.splitext(os.path.basename(zip_path))[0])
    os.makedirs(out_dir, exist_ok=True)
    extracted = []

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.namelist():
                if not member.lower().endswith(".nc"):
                    continue

                safe = os.path.normpath(member)
                if safe.startswith("..") or os.path.isabs(safe):
                    continue

                target = os.path.join(out_dir, safe)
                os.makedirs(os.path.dirname(target), exist_ok=True)

                if not os.path.exists(target):
                    with zf.open(member) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)

                extracted.append(target)
    except Exception as e:
        print("[WARN] ZIP:", zip_path, "|", e)

    return sorted(set(extracted))

zip_nc = []
for zp in zip_paths:
    low = os.path.basename(zp).lower()
    if ".safe" in low or "sentinel" in low:
        print("Skipping optical/Sentinel archive:", zp)
        continue
    zip_nc.extend(extract_nc_from_zip(zp, SCRATCH))

all_nc = sorted(set(direct_nc + zip_nc))
print("Total NC candidates:", len(all_nc))
for p in all_nc:
    print(" -", p)

In [ ]:
def nc_info(path):
    try:
        ds = xr.open_dataset(path, decode_times=True)

        lat_name = "latitude" if "latitude" in ds.coords else ("lat" if "lat" in ds.coords else None)
        lon_name = "longitude" if "longitude" in ds.coords else ("lon" if "lon" in ds.coords else None)

        if lat_name and lon_name:
            lat_min = float(ds[lat_name].min())
            lat_max = float(ds[lat_name].max())
            lon_min = float(ds[lon_name].min())
            lon_max = float(ds[lon_name].max())
        else:
            lat_min = lat_max = lon_min = lon_max = np.nan

        times = pd.to_datetime(ds["time"].values) if "time" in ds.coords else pd.DatetimeIndex([])
        info = {
            "vars": set(ds.data_vars),
            "lat_min": lat_min,
            "lat_max": lat_max,
            "lon_min": lon_min,
            "lon_max": lon_max,
            "time_min": str(times.min())[:10] if len(times) else "?",
            "time_max": str(times.max())[:10] if len(times) else "?",
            "ntime": len(times),
        }
        ds.close()
        return info
    except Exception as e:
        print("[WARN]", os.path.basename(path), e)
        return None

infos = {}
for p in all_nc:
    x = nc_info(p)
    if x is not None:
        infos[p] = x

# The four newly downloaded IO files.
io_my = [
    p for p, info in infos.items()
    if "_my_0.083deg_p1d" in os.path.basename(p).lower()
    and set(DYNAMIC_VARS).issubset(info["vars"])
    and (
        "indian ocean" in p.lower()
        or "new indian ocean" in p.lower()
        or "indian_ocean" in p.lower()
        or "indianocean" in p.lower()
    )
]

# Fallback if the folder has a different name: identify a full IO MY footprint.
if not io_my:
    io_my = [
        p for p, info in infos.items()
        if "_my_0.083deg_p1d" in os.path.basename(p).lower()
        and set(DYNAMIC_VARS).issubset(info["vars"])
        and info["lat_min"] <= -19
        and info["lat_max"] >= 24
        and info["lon_min"] <= 41
        and info["lon_max"] >= 99
    ]

if not io_my:
    raise FileNotFoundError(
        "The new IO MY files were not found. Put the four downloaded .nc files "
        "inside AquaStrat/New Indian Ocean (or a subfolder under AquaStrat)."
    )

def overlaps_region(path, region):
    info = infos[path]
    g = master[master["region"] == region]
    if g.empty:
        return False
    return (
        info["lat_min"] <= g["lat_center"].max()
        and info["lat_max"] >= g["lat_center"].min()
        and info["lon_min"] <= g["lon_center"].max()
        and info["lon_max"] >= g["lon_center"].min()
    )

as_candidates = []
bob_candidates = []

for p, info in infos.items():
    low = p.lower()
    if p in io_my:
        continue
    if "indian ocean" in low or "new indian ocean" in low:
        continue

    if "arabian" in low or os.path.basename(p).lower() == "ocean_physics.nc":
        as_candidates.append(p)
    elif "bengal" in low or "bob" in low:
        bob_candidates.append(p)

if not as_candidates:
    as_candidates = [p for p in infos if p not in io_my and overlaps_region(p, "AS")]

if not bob_candidates:
    bob_candidates = [p for p in infos if p not in io_my and overlaps_region(p, "BOB")]

as_candidates = sorted(set(as_candidates))
bob_candidates = sorted(set(bob_candidates))
io_my = sorted(set(io_my))

print("\n=== SELECTED ===")
print("AS:")
for p in as_candidates:
    print(" ", os.path.basename(p), infos[p]["vars"], infos[p]["time_min"], "to", infos[p]["time_max"])
print("BOB:")
for p in bob_candidates:
    print(" ", os.path.basename(p), infos[p]["vars"], infos[p]["time_min"], "to", infos[p]["time_max"])
print("IO MY:")
for p in io_my:
    print(" ", os.path.basename(p), infos[p]["vars"], infos[p]["time_min"], "to", infos[p]["time_max"])

In [ ]:
# ---- copied unchanged (Step 4: open regional datasets) ----
def open_nc_group(paths, label):
    paths = sorted(set(paths))
    if not paths:
        return None

    print(f"[{label}] opening {len(paths)} file(s)")

    if len(paths) == 1:
        ds = xr.open_dataset(paths[0], chunks={})
    else:
        try:
            ds = xr.open_mfdataset(
                paths,
                combine="by_coords",
                chunks={},
                parallel=False,
                compat="override",
                coords="minimal",
                data_vars="minimal",
                join="outer",
            )
        except Exception as e:
            print(f"[{label}] open_mfdataset failed:", e)
            parts = [xr.open_dataset(p, chunks={}) for p in paths]
            ds = xr.concat(parts, dim="time").sortby("time")

    if "depth" in ds.dims and ds.sizes["depth"] == 1:
        ds = ds.squeeze("depth", drop=True)

    keep = [v for v in DYNAMIC_VARS if v in ds.data_vars]
    ds = ds[keep]

    if "time" in ds.coords:
        t = pd.to_datetime(ds.time.values)
        _, idx = np.unique(t, return_index=True)
        if len(idx) != len(t):
            ds = ds.isel(time=np.sort(idx))
        ds = ds.sortby("time")

    print(f"[{label}] dims:", dict(ds.sizes))
    print(f"[{label}] vars:", list(ds.data_vars))

    if "time" in ds.coords:
        t = pd.to_datetime(ds.time.values)
        print(f"[{label}] {str(t.min())[:10]} -> {str(t.max())[:10]} ({len(t)} days)")

    return ds

datasets = {
    "AS": open_nc_group(as_candidates, "AS"),
    "BOB": open_nc_group(bob_candidates, "BOB"),
    "IO": open_nc_group(io_my, "IO-MY"),
}

In [ ]:
# ---- copied unchanged (Step 5: regional series) ----
def coord_name(ds, names):
    for n in names:
        if n in ds.coords:
            return n
    return None

def fill_nan_rows(arr):
    out = np.asarray(arr, dtype=np.float32)
    for i in range(out.shape[0]):
        s = pd.Series(out[i])
        out[i] = s.interpolate(limit_direction="both").to_numpy(dtype=np.float32)
    return out

def select_grid_cells(grid, ds):
    lat_name = coord_name(ds, ["latitude", "lat"])
    lon_name = coord_name(ds, ["longitude", "lon"])

    if lat_name is None or lon_name is None:
        raise ValueError(f"Cannot find lat/lon in {list(ds.coords)}")

    lats = np.asarray(ds[lat_name].values, dtype=float)
    lons = np.asarray(ds[lon_name].values, dtype=float)

    lat_step = float(np.median(np.abs(np.diff(np.unique(lats))))) if len(lats) > 1 else 0.1
    lon_step = float(np.median(np.abs(np.diff(np.unique(lons))))) if len(lons) > 1 else 0.1

    lat_min, lat_max = lats.min() - lat_step/2, lats.max() + lat_step/2
    lon_min, lon_max = lons.min() - lon_step/2, lons.max() + lon_step/2

    keep = (
        grid["lat_center"].between(lat_min, lat_max)
        & grid["lon_center"].between(lon_min, lon_max)
        & (~grid["is_land"].astype(bool))
    )

    g = grid.loc[keep].reset_index(drop=True)
    if g.empty:
        raise ValueError("No master-grid cells are covered by the real dataset.")
    return g, lat_name, lon_name

def build_region(region, ds):
    grid = master[master["region"] == region].copy().reset_index(drop=True)
    grid, lat_name, lon_name = select_grid_cells(grid, ds)

    lat_da = xr.DataArray(grid["lat_center"].to_numpy(), dims="cell")
    lon_da = xr.DataArray(grid["lon_center"].to_numpy(), dims="cell")

    selected = ds.sel(
        {lat_name: lat_da, lon_name: lon_da},
        method="nearest",
    )

    series = {}
    for v in DYNAMIC_VARS:
        if v in selected.data_vars:
            series[v] = fill_nan_rows(np.asarray(selected[v].values).T)

    if not {"uo", "vo"}.issubset(series):
        raise ValueError(f"{region}: both uo and vo are required.")

    valid = np.ones(len(grid), dtype=bool)
    for arr in series.values():
        valid &= ~np.isnan(arr).any(axis=1)

    grid = grid.loc[valid].reset_index(drop=True)
    for v in series:
        series[v] = series[v][valid]

    n_cells, n_days = series["uo"].shape
    dates = pd.to_datetime(ds.time.values)
    doy = dates.dayofyear.to_numpy(dtype=np.float32)

    wildlife = grid["wildlife_exposure"].to_numpy(dtype=np.float32)
    habitat = grid["habitat_sensitivity"].to_numpy(dtype=np.float32)
    coastal = (0.5 * wildlife + 0.5 * habitat).reshape(-1, 1)

    monsoon = np.exp(
        -((doy - 200.0) ** 2) / (2.0 * 55.0 ** 2)
    ).reshape(1, -1)

    current_mag = np.sqrt(series["uo"]**2 + series["vo"]**2)
    cmin, cmax = current_mag.min(), current_mag.max()
    current_norm = (current_mag - cmin) / (cmax - cmin + 1e-9)

    base = (
        0.15
        + 0.5 * coastal
        * (1.0 + 1.2 * monsoon)
        * (1.0 + 0.4 * current_norm)
    )

    rng = np.random.default_rng(42 + {"AS": 1, "BOB": 2, "IO": 3}[region])
    noise = rng.normal(0, 0.05, size=(n_cells, n_days)).astype(np.float32)
    noise = (
        pd.DataFrame(noise.T)
        .rolling(5, min_periods=1, center=True)
        .mean()
        .to_numpy()
        .T
        .astype(np.float32)
    )

    density = np.clip(base + noise, 0, None)
    density = (
        (density - density.min())
        / (density.max() - density.min() + 1e-9)
    ).astype(np.float32)

    series["density"] = density

    return {
        "grid": grid,
        "series": series,
        "n_cells": n_cells,
        "n_days": n_days,
        "day_of_year": doy,
    }

region_data = {}
for r in REGIONS:
    if datasets[r] is None:
        raise RuntimeError(f"{r} has no real ocean-physics dataset.")
    region_data[r] = build_region(r, datasets[r])
    d = region_data[r]
    print(f"{r}: REAL DAILY | cells={d['n_cells']} | days={d['n_days']} | vars={list(d['series'])}")

In [ ]:
# ---- Step 6 of the original, but KEEPING the mean/std it used ----
BASE_FEATURES = ["thetao","so","uo","vo","zos","wildlife_exposure","habitat_sensitivity",
                 "lat_center","lon_center","sin_doy","cos_doy","current_mag"]

STATIC_MAP = {"thetao": "sst", "so": "salinity", "uo": "current_u", "vo": "current_v", "zos": "ssh"}
PRESENT = {r: [v in region_data[r]["series"] for v in DYNAMIC_VARS] for r in REGIONS}   # what training actually had
print("Dynamic vars present in training data:", PRESENT)

def make_raw_features(g, dyn, doy, region):
    """g: grid rows, dyn: dict var -> (cells, days) raw arrays, doy: day-of-year per day"""
    n, t = dyn["uo"].shape
    feats = np.zeros((n, t, len(BASE_FEATURES)), dtype=np.float32)
    for j, v in enumerate(DYNAMIC_VARS):
        if PRESENT[region][j]:
            feats[:, :, j] = dyn[v]
        else:   # same static fallback the training notebook used
            gm = g[["grid_id"]].merge(master[["grid_id", STATIC_MAP[v]]], on="grid_id", how="left")
            feats[:, :, j] = np.nan_to_num(gm[STATIC_MAP[v]].to_numpy(np.float32), nan=0.0)[:, None]
    feats[:, :, 5] = g["wildlife_exposure"].to_numpy(np.float32)[:, None]
    feats[:, :, 6] = g["habitat_sensitivity"].to_numpy(np.float32)[:, None]
    feats[:, :, 7] = g["lat_center"].to_numpy(np.float32)[:, None]
    feats[:, :, 8] = g["lon_center"].to_numpy(np.float32)[:, None]
    feats[:, :, 9] = np.sin(2*np.pi*doy/365.25).astype(np.float32)[None, :]
    feats[:, :, 10] = np.cos(2*np.pi*doy/365.25).astype(np.float32)[None, :]
    feats[:, :, 11] = np.sqrt(feats[:, :, 2]**2 + feats[:, :, 3]**2)
    return feats

STATS = {}
for region in REGIONS:
    d = region_data[region]
    feats = make_raw_features(d["grid"], d["series"], d["day_of_year"], region)
    train_end = int(d["n_days"] * 0.70)
    mu = np.array([np.nanmean(feats[:, :train_end, j]) for j in range(feats.shape[-1])], np.float64)
    sd = np.array([np.nanstd(feats[:, :train_end, j]) for j in range(feats.shape[-1])], np.float64)
    STATS[region] = (mu, sd)
    d["feats_raw"] = feats
    print(region, "train_end =", train_end, "| mu[:5] =", np.round(mu[:5], 3))

ONEHOT = {r: np.eye(len(REGIONS), dtype=np.float32)[i] for i, r in enumerate(REGIONS)}

def to_model_input(feats_raw, region):
    mu, sd = STATS[region]
    x = ((feats_raw - mu) / (sd + 1e-8)).astype(np.float32)
    masks = np.broadcast_to(np.array(PRESENT[region], np.float32), x.shape[:-1] + (len(DYNAMIC_VARS),))
    oh = np.broadcast_to(ONEHOT[region], x.shape[:-1] + (len(REGIONS),))
    return np.concatenate([x, masks, oh], axis=-1)

# ---- VERIFY: reproduce the saved v5 predictions from the last 14 days of 2025 ----
rows = []
for region in REGIONS:
    d = region_data[region]
    x = to_model_input(d["feats_raw"][:, -WINDOW:, :], region)
    p = model.predict(x, batch_size=4096, verbose=0).reshape(-1)
    rows.append(pd.DataFrame({"grid_id": d["grid"]["grid_id"].values, "repro": p}))
chk = pd.concat(rows).merge(v5[["grid_id", "predicted_density"]], on="grid_id")
err = np.abs(chk["repro"] - chk["predicted_density"])
print(f"\nVERIFICATION vs predicted_hotspots_v5.csv: cells={len(chk)}  mean|diff|={err.mean():.6f}  max|diff|={err.max():.6f}  corr={np.corrcoef(chk.repro, chk.predicted_density)[0,1]:.5f}")
VERIFIED = err.mean() < 1e-3
print("✅ Pipeline reproduces your v5 model output exactly." if VERIFIED else "⚠ Not an exact match — keep going, the zip still gets made; just send a screenshot of this line too.")
V5_IDS = set(v5.grid_id)   # keep exactly the 14,777 cells the dashboard uses
CELLS = {r: region_data[r]["grid"].loc[region_data[r]["grid"].grid_id.isin(V5_IDS), ["grid_id", "lat_center", "lon_center", "wildlife_exposure", "habitat_sensitivity"]].reset_index(drop=True) for r in REGIONS}
print("Cells used per region:", {r: len(CELLS[r]) for r in REGIONS})
# ---- keep a small reference sample (last 14 days of 2025) so the GitHub automation can re-verify the model every day ----
REF = {}
for r in REGIONS:
    d = region_data[r]; gid = d["grid"]["grid_id"].values
    idx = np.where(np.isin(gid, list(V5_IDS)))[0][::max(1, len(CELLS[r]) // 150)][:150]
    v5map = dict(zip(v5.grid_id, v5.predicted_density))
    REF[r] = {"grid_id": [str(x) for x in gid[idx]],
              "raw": np.round(d["feats_raw"][idx, -WINDOW:, :].astype(np.float64), 7).tolist(),
              "v5": [float(v5map[x]) for x in gid[idx]]}
for r in REGIONS:
    region_data[r].pop("feats_raw", None)
for ds_ in datasets.values():
    try: ds_.close()
    except Exception: pass

## Step 4 — Log in to **Copernicus Marine** (marine.copernicus.eu)
⚠ This is NOT the Copernicus Data Space / Copernicus Browser login (dataspace.copernicus.eu) — that is a different service.
Fill in your **e-mail** and password (Copernicus Marine accepts the e-mail instead of the username).
No Marine account yet? Free: https://data.marine.copernicus.eu/register — or open data.marine.copernicus.eu once and sign in with your Copernicus Data Space account.

In [ ]:
#@title Copernicus Marine login { display-mode: "form" }
CM_EMAIL = ""  #@param {type:"string"}
CM_PASSWORD = ""  #@param {type:"string"}
user = CM_EMAIL.strip() or input("Copernicus Marine e-mail: ").strip()
pwd = CM_PASSWORD or getpass.getpass("Copernicus Marine password: ")
try:
    ok = cm.login(username=user, password=pwd, force_overwrite=True, check_credentials_valid=True)
except Exception as e:
    ok = False; print("Login error:", e)
if not ok:
    raise SystemExit("❌ Copernicus MARINE login failed. If you used your Copernicus Browser / Data Space login, it will not work here — "
                     "create a free account at https://data.marine.copernicus.eu/register (or use the account that downloaded your 2025 NetCDF data), "
                     "then put the e-mail/password in the form above and re-run this cell.")
CM_U, CM_P = user, pwd   # reused for every download below — no more prompts
import logging; logging.getLogger("copernicusmarine").setLevel(logging.ERROR)
print("✅ Logged in to Copernicus Marine as", user)

## Step 5 — Download the latest real ocean physics (Global Analysis & Forecast, 1/12°, daily)
Only the surface layer and only the bounding box of each region; values are sampled at your exact grid-cell centres (same as training).

In [ ]:
DATASETS = {
    "thetao": ["cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m"],
    "so":     ["cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m"],
    "uo":     ["cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m"],
    "vo":     ["cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m"],
    "zos":    ["cmems_mod_glo_phy_anfc_0.083deg_P1D-m"],
}
START = (TODAY - pd.Timedelta(days=PAST_DAYS)).strftime("%Y-%m-%d")
END = (TODAY + pd.Timedelta(days=FUTURE_DAYS)).strftime("%Y-%m-%d")

def fetch(var, region):
    g = CELLS[region]
    last = None
    for ds_id in DATASETS[var]:
        try:
            ds = cm.open_dataset(
                dataset_id=ds_id, variables=[var], username=CM_U, password=CM_P,
                minimum_longitude=float(g.lon_center.min()) - 0.3, maximum_longitude=float(g.lon_center.max()) + 0.3,
                minimum_latitude=float(g.lat_center.min()) - 0.3, maximum_latitude=float(g.lat_center.max()) + 0.3,
                start_datetime=START, end_datetime=END,
                **({} if var == "zos" else {"minimum_depth": 0, "maximum_depth": 1}),
            )
            if "depth" in ds[var].dims:
                ds = ds.isel(depth=0)
            lat = xr.DataArray(g.lat_center.values, dims="cell")
            lon = xr.DataArray(g.lon_center.values, dims="cell")
            da = ds[var].sel(latitude=lat, longitude=lon, method="nearest").load()
            return pd.to_datetime(da.time.values).normalize(), np.asarray(da.values, np.float32).T  # (cells, days)
        except Exception as e:
            last = e
            print(f"   [{var}/{region}] {ds_id} failed: {str(e)[:200]}")
    raise RuntimeError(f"Could not download {var} for {region}: {last}")

NEW = {}
for region in REGIONS:
    print(f"== {region}: {len(CELLS[region])} cells, {START} → {END}")
    got = {}
    for v in DYNAMIC_VARS:
        t, a = fetch(v, region)
        got[v] = pd.DataFrame(a, columns=t)
        print(f"   {v}: {a.shape[1]} days ({t.min().date()} → {t.max().date()})")
    common = sorted(set.intersection(*[set(df.columns) for df in got.values()]))
    dyn = {}
    for v in DYNAMIC_VARS:
        arr = got[v][common].to_numpy(np.float32)
        arr = pd.DataFrame(arr.T).interpolate(limit_direction="both").to_numpy(np.float32).T  # same gap-fill as training
        arr = np.where(np.isnan(arr), np.nanmean(arr), arr)
        dyn[v] = arr
    NEW[region] = {"dates": pd.DatetimeIndex(common), "dyn": dyn}
    print(f"   common days: {len(common)} ({common[0].date()} → {common[-1].date()})")

## Step 6 — Run the trained LSTM for every cell and every day

In [ ]:
out = []
for region in REGIONS:
    g, D = CELLS[region], NEW[region]
    dates, n_days = D["dates"], len(D["dates"])
    feats = make_raw_features(g, D["dyn"], dates.dayofyear.to_numpy(np.float32), region)
    x_all = to_model_input(feats, region)
    for e in range(WINDOW - 1, n_days):                    # window ends on dates[e] -> predicts the next day
        target = dates[e] + pd.Timedelta(days=HORIZON)
        p = model.predict(x_all[:, e - WINDOW + 1:e + 1, :], batch_size=4096, verbose=0).reshape(-1)
        out.append(pd.DataFrame({
            "date": target.strftime("%Y-%m-%d"), "grid_id": g.grid_id.values, "region": region,
            "predicted_density": np.round(p, 5),
            "physics": "analysis" if dates[e] <= TODAY else "forecast",
        }))
    print(f"{region}: predicted {n_days - WINDOW + 1} days")
pred = pd.concat(out, ignore_index=True)
print(pred.groupby(["region"]).date.agg(["min", "max", "nunique"]))
summary = pred.pivot_table(index="date", columns="region", values="predicted_density", aggfunc="mean").round(4)
summary

## Step 7 — Quick check map (latest day) + save & download

In [ ]:
import matplotlib.pyplot as plt
last = pred.date.max()
m = pred[pred.date == last].merge(master[["grid_id", "lat_center", "lon_center"]], on="grid_id")
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
for a, r in zip(ax, REGIONS):
    s = m[m.region == r]
    sc = a.scatter(s.lon_center, s.lat_center, c=s.predicted_density, s=6, cmap="turbo")
    a.set_title(f"{r} · {last}"); plt.colorbar(sc, ax=a)
plt.tight_layout(); plt.show()

stamp = TODAY.strftime("%Y%m%d")
OUT_DIR = f"{BASE}/lstm_daily"; os.makedirs(OUT_DIR, exist_ok=True)
csv_path = f"/content/aquastrat_daily_predictions_{stamp}.csv"
pred.to_csv(csv_path, index=False)
meta = pd.DataFrame([{
    "run_date": TODAY.date(), "model": "aqua_strat_lstm_v5.keras", "verified_vs_v5": VERIFIED,
    "first_date": pred.date.min(), "last_date": pred.date.max(), "cells": pred.grid_id.nunique(),
    "physics_source": "Copernicus GLOBAL_ANALYSISFORECAST_PHY_001_024 (daily, 1/12 deg, surface)",
}])
meta_path = f"/content/aquastrat_daily_meta_{stamp}.csv"; meta.to_csv(meta_path, index=False)
zip_path = f"/content/aquastrat_daily_predictions_{stamp}.zip"
# Also pack your real FDI outputs (from AQUA_STRAT_FDI_Final.ipynb) if they are on Drive
fdi_files = []
for name in ["fdi_density_validation.csv", "fdi_per_tile_summary.csv"]:
    hits = sorted(glob.glob("/content/drive/MyDrive/**/" + name, recursive=True), key=len)
    if hits: fdi_files.append(hits[0])
print("FDI artifacts found:", fdi_files or "none (FDI page will show notebook results only)")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, os.path.basename(csv_path)); z.write(meta_path, os.path.basename(meta_path))
    for p in fdi_files: z.write(p, os.path.basename(p))
shutil.copy2(zip_path, OUT_DIR)
print("Saved to Drive:", OUT_DIR, "|", round(os.path.getsize(zip_path)/1e6, 1), "MB")
print(meta.T)
from google.colab import files
files.download(zip_path)

## Step 8 — (ONE TIME) Export files for the automatic daily GitHub run
Leave `EXPORT_FOR_GITHUB = True` the first time. It downloads **`model_inputs.json`** (training normalisation, the 14,777 grid cells and a small verification sample — no passwords).
Put that file in the **`pipeline/`** folder of the GitHub repository. After that, GitHub Actions refreshes the forecast every morning by itself and this notebook is only needed as a backup.

In [ ]:
import json
EXPORT_FOR_GITHUB = True  #@param {type:"boolean"}
if EXPORT_FOR_GITHUB:
    exp = {
        "about": "AQUA-STRAT LSTM v5 inference inputs for pipeline/daily_forecast.py (no credentials inside)",
        "created": str(TODAY.date()), "window": WINDOW, "horizon": HORIZON,
        "features": BASE_FEATURES, "dynamic_vars": DYNAMIC_VARS, "regions": REGIONS,
        "verified_vs_v5": bool(VERIFIED),
        "stats": {r: {"mu": [float(x) for x in STATS[r][0]], "sd": [float(x) for x in STATS[r][1]]} for r in REGIONS},
        "present": {r: [bool(x) for x in PRESENT[r]] for r in REGIONS},
        "cells": {r: {k: (CELLS[r][k].astype(str).tolist() if k == "grid_id" else [float(x) for x in CELLS[r][k]])
                      for k in ["grid_id", "lat_center", "lon_center", "wildlife_exposure", "habitat_sensitivity"]} for r in REGIONS},
        "reference": REF,
    }
    with open("/content/model_inputs.json", "w") as f: json.dump(exp, f)
    print("model_inputs.json:", round(os.path.getsize("/content/model_inputs.json")/1e6, 2), "MB | cells:", {r: len(CELLS[r]) for r in REGIONS})
    shutil.copy2("/content/model_inputs.json", BASE)
    from google.colab import files
    files.download("/content/model_inputs.json")
